In [2]:
import os

DATA = "data"

print(os.listdir(DATA))
print(os.listdir(DATA + "/train"))
print(os.listdir(DATA + "/valid"))
print(os.listdir(DATA + "/test"))

['test', 'train', 'valid']
['-1x-1_jpg.rf.8d697c00929e06e6655c08835cc66c02.jpg', '-I1-MS09uaqsLdGTFkgnS0Rcg1mmPyAj95ySg_eckoM_jpeg.rf.8f09ebde8b7b3ea6f9180eff345ec176.jpg', '0002526673_jpg.rf.2c547115c01c1b5a10c6d467551f0cae.jpg', '000b7b75-1600_jpg.rf.5d7117e8571505dbfe49e2f737089ea0.jpg', '000_1ov3n5_0_jpeg.rf.a23f1c89491779996f4519858277a4e0.jpg', '001_1024_jpeg.rf.f915f4689737658b59732df33fdbee22.jpg', '003_1024_jpeg.rf.bc025d99896124ddce6fa706b4a5c7b3.jpg', '004_1024_jpeg.rf.763d99dadfb9fe5f3f72ff3cc34f88b6.jpg', '012106_jpg_1140x855_jpg.rf.c6524deb426da2c246a03ac0e610f0cb.jpg', '012420_coronoa_masks_web_jpg.rf.dcc655a29611260a439d59c13b0cad1a.jpg', '0200b38c89b16c37c5de8e247bb00c2f_jpg.rf.6a5b142fd82320d571402836bce2cf0a.jpg', '0209-00176-076b1_jpg.rf.e9d992f7258f2c5ae8c29f76a727efda.jpg', '0450908675_50159485_mutation-virus-chine-inquietude_jpg.rf.466d55e174c1bc43889c49b06aad0d75.jpg', '0602623232127-web-tete_jpg.rf.70c23b5e96669fb4f31a7cbd6d52a670.jpg', '0_10725_jpg.rf.99ff78c8

In [4]:
import os
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report
import time

IMG = 128
BATCH = 16
DATA = "data"


# -------------------------------
# Load images and labels
# -------------------------------
def load_data(folder):

    df = pd.read_csv(os.path.join(folder, "_annotations.csv"))

    # Group annotations belonging to the same image
    labels = df.groupby("filename")["class"].apply(
        lambda x: 1 if all(x == "mask") else 0
    )

    paths = [
        os.path.join(folder, f)
        for f in labels.index
    ]

    return paths, labels.values


train_paths, train_labels = load_data(DATA + "/train")
val_paths, val_labels = load_data(DATA + "/valid")
test_paths, test_labels = load_data(DATA + "/test")

print("Train:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))


# -------------------------------
# Image preprocessing
# -------------------------------
def preprocess(path, label):

    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG, IMG))
    img = tf.cast(img, tf.float32) / 255.0

    return img, label


train = tf.data.Dataset.from_tensor_slices(
    (train_paths, train_labels)
).map(preprocess).batch(BATCH)

val = tf.data.Dataset.from_tensor_slices(
    (val_paths, val_labels)
).map(preprocess).batch(BATCH)

test = tf.data.Dataset.from_tensor_slices(
    (test_paths, test_labels)
).map(preprocess).batch(BATCH)


# -------------------------------
# MobileNetV2
# -------------------------------
base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG, IMG, 3),
    include_top=False,
    weights="imagenet"
)

base.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),

    base,

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation="sigmoid")
])


model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# -------------------------------
# Training
# -------------------------------
model.fit(
    train,
    validation_data=val,
    epochs=3
)


# -------------------------------
# Evaluation
# -------------------------------
y_true = []
y_pred = []

for x, y in test:

    p = model.predict(x, verbose=0).ravel()

    y_true.extend(y.numpy())
    y_pred.extend((p > 0.5).astype(int))


print("\nClassification Report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=["No Mask", "Mask"]
    )
)


# -------------------------------
# Inference speed
# -------------------------------
x = next(iter(test))[0]

start = time.time()

for _ in range(100):
    model(x, training=False)

print(
    "\nInference time:",
    (time.time() - start) / 100,
    "seconds/batch"
)


# -------------------------------
# TensorFlow Lite
# -------------------------------
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

tflite_model = converter.convert()

with open("mask_detector.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved!")

Train: 105
Validation: 29
Test: 15
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Epoch 1/3
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 250ms/step - accuracy: 0.5143 - loss: 0.8803 - val_accuracy: 0.5862 - val_loss: 0.6474
Epoch 2/3
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.5524 - loss: 0.7357 - val_accuracy: 0.6552 - val_loss: 0.6437
Epoch 3/3
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.6476 - loss: 0.7587 - val_accuracy: 0.6552 - val_loss: 0.6431

Classification Report:

              precision    recall  f1-score   support

     No Mask       0.00      0.00      0.00         3
        Mask       0.75      0.75      0.75        12

    accuracy                           0.60        15
   macro avg       0.38      0.38      0.38        15
weighted avg       0.60      0.60      0.60        15


Inference time: 0.2133057975769043 seconds/batch
INFO:tensorflow:Assets written to: C:\Users\TEJESW~1\AppData\Local\Temp\tmpatr1ua3v\assets


INFO:tensorflow:Assets written to: C:\Users\TEJESW~1\AppData\Local\Temp\tmpatr1ua3v\assets


Saved artifact at 'C:\Users\TEJESW~1\AppData\Local\Temp\tmpatr1ua3v'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2456496647312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2456496647696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2456496647120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2456496649040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2456496648080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2456496647504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2456496649424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2456496650192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2456496649808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2456496647888: TensorSpec(shape=(), dtype=tf.resource, name=None)
